# Bibliotecas

In [ ]:
import sys
import os

sys.path.append(os.path.abspath('../'))

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import time
import json

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split

from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

import src.futurai_ppd as ppd
import src.futurai_utils as utils

# from src.MSCRED import MSCRED

from src.MSCVAE_ORIGINAL import MSCVAE
# from src.MSCVAE_ELBOW import MSCVAE
# from src.MSCVAE_MAD import MSCVAE
# from src.MSCVAE_GRADIENT import MSCVAE

DIR_DATA = "/Users/falcone/Desktop/github-repos/anomaly-detection/data/"

# Parâmetros gerais e dados do processo

In [ ]:
base_name = "392P2007 Bomba.csv"
timestamp = "Timestamp"
process_name = "392P2007 Bomba"
company_name = "Klabin - MA"
subsistema = DIR_DATA+ process_name +'_subsistema.csv'

# Carregando dados

In [ ]:
list_colomuns_drop = ['392M3820_ST']

df_dataset = ppd.load_dataset_principal(DIR_DATA+process_name+".csv", list_colomuns_drop, timestamp, dropna=True, use_chunks=True, chunksize=10000)
df_dataset

# Parâmetros de ON/OFF

In [ ]:
pre_process = []

pp_var_ref_desligado = '392P2007_ST'
pp_valor_ref_desligado = 400
pp_tempo_ref_desligado = 0
pp_pre_corte_transitorio = 90
pp_pos_corte_transitorio = 150
pre_process.append(  
{
   "after_cut": pp_pos_corte_transitorio,
   "interval_off": pp_tempo_ref_desligado,
   "limit_off": pp_valor_ref_desligado,
   "pre_cut": pp_pre_corte_transitorio,
   "variable_off": pp_var_ref_desligado
  })

# Importação das tags e descrições

In [ ]:
df_sistema, df_sistema_drop =  ppd.set_tags_config(df_dataset,subsistema)
df_sistema

In [ ]:
df_sistema_drop

# Seleção do período de treinamento e criação do modelo

In [ ]:
df_train_list = []

## Período 1

In [ ]:
start_date_train = pd.to_datetime('2025-09-26 00:00:00')
end_date_train = pd.to_datetime('2025-09-30 00:00:00')

mask = (df_dataset[timestamp] >= start_date_train) & (df_dataset[timestamp] <= end_date_train)
df_train = df_dataset.loc[mask]

# pre - processamento
print(df_train.shape)

list_periods_train = []
for pro in pre_process:
    df_train,_,list_aux = ppd.drop_transitorio_desligado(df_train,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_train = [*list_periods_train,*list_aux]

df_train.reset_index(inplace=True, drop=True)

print(df_train.shape)

eixoX_train = df_train[timestamp]
df_train = df_train.drop(timestamp,axis=1)
df_train1 = df_train.copy()

df_train_list.append(df_train1)

## Período 2

In [ ]:
start_date_train2 = pd.to_datetime('2025-12-12 00:00:00')
end_date_train2 = pd.to_datetime('2025-12-12 04:00:00')


mask = (df_dataset[timestamp] > start_date_train2) & (df_dataset[timestamp] <= end_date_train2)
df_train2 = df_dataset.loc[mask]

# pre - processamento
print(df_train2.shape)

list_periods_train = []
for pro in pre_process:
    df_train2,_,list_aux = ppd.drop_transitorio_desligado(df_train2,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_train = [*list_periods_train,*list_aux]

df_train2.reset_index(inplace=True, drop=True)

print(df_train2.shape)


eixoX_train2 = df_train2[timestamp]
df_train2 = df_train2.drop(timestamp,axis=1)

df_train = pd.concat([df_train, df_train2], ignore_index=True)
eixoX_train = pd.concat([eixoX_train, eixoX_train2], ignore_index=True)

df_train_list.append(df_train2)

## Período 3

In [ ]:
#Data train 
start_date_train3 = pd.to_datetime('2026-01-06 12:00:00') 
end_date_train3 = pd.to_datetime('2026-01-07 00:00:00')

mask = (df_dataset[timestamp] > start_date_train3) & (df_dataset[timestamp] <= end_date_train3)
df_train3 = df_dataset.loc[mask]

# pre - processamento
print(df_train3.shape)

list_periods_train = []
for pro in pre_process:
    df_train3,_,list_aux = ppd.drop_transitorio_desligado(df_train3,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_train = [*list_periods_train,*list_aux]

df_train3.reset_index(inplace=True, drop=True)

print(df_train3.shape)


eixoX_train3 = df_train3[timestamp]
df_train3 = df_train3.drop(timestamp,axis=1)

df_train = pd.concat([df_train, df_train3], ignore_index=True)
eixoX_train = pd.concat([eixoX_train, eixoX_train3], ignore_index=True)

df_train_list.append(df_train3)

## Período 4

In [ ]:
start_date_train4 = pd.to_datetime('2026-01-10 06:00:00') 
end_date_train4 = pd.to_datetime('2026-01-10 18:00:00')

mask = (df_dataset[timestamp] > start_date_train4) & (df_dataset[timestamp] <= end_date_train4)
df_train4 = df_dataset.loc[mask]

# pre - processamento
print(df_train4.shape)

list_periods_train = []
for pro in pre_process:
    df_train4,_,list_aux = ppd.drop_transitorio_desligado(df_train4,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_train = [*list_periods_train,*list_aux]

df_train4.reset_index(inplace=True, drop=True)

print(df_train4.shape)


eixoX_train4 = df_train4[timestamp]
df_train4 = df_train4.drop(timestamp,axis=1)

df_train = pd.concat([df_train, df_train4], ignore_index=True)
eixoX_train = pd.concat([eixoX_train, eixoX_train4], ignore_index=True)

df_train_list.append(df_train4)

## Período 5

In [ ]:
start_date_train5 = pd.to_datetime('2026-01-23 06:00:00') 
end_date_train5 = pd.to_datetime('2026-01-24 20:00:00')

mask = (df_dataset[timestamp] > start_date_train5) & (df_dataset[timestamp] <= end_date_train5)
df_train5 = df_dataset.loc[mask]

# pre - processamento
print(df_train5.shape)

list_periods_train = []
for pro in pre_process:
    df_train5,_,list_aux = ppd.drop_transitorio_desligado(df_train5,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_train = [*list_periods_train,*list_aux]

df_train5.reset_index(inplace=True, drop=True)

print(df_train5.shape)


eixoX_train5 = df_train5[timestamp]
df_train5 = df_train5.drop(timestamp,axis=1)

df_train = pd.concat([df_train, df_train5], ignore_index=True)
eixoX_train = pd.concat([eixoX_train, eixoX_train5], ignore_index=True)

df_train_list.append(df_train5)

## Período 6

In [ ]:
start_date_train6 = pd.to_datetime('2026-02-11 22:00:00') 
end_date_train6 = pd.to_datetime('2026-02-12 00:00:00')

mask = (df_dataset[timestamp] > start_date_train6) & (df_dataset[timestamp] <= end_date_train6)
df_train6 = df_dataset.loc[mask]

# pre - processamento
print(df_train6.shape)

list_periods_train = []
for pro in pre_process:
    df_train6,_,list_aux = ppd.drop_transitorio_desligado(df_train6,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_train = [*list_periods_train,*list_aux]

df_train6.reset_index(inplace=True, drop=True)

print(df_train6.shape)


eixoX_train6 = df_train6[timestamp]
df_train6 = df_train6.drop(timestamp,axis=1)

df_train = pd.concat([df_train, df_train6], ignore_index=True)
eixoX_train = pd.concat([eixoX_train, eixoX_train6], ignore_index=True)

df_train_list.append(df_train6)

## Período 7

In [ ]:
start_date_train7 = pd.to_datetime('2026-03-02 10:00:00') 
end_date_train7 = pd.to_datetime('2026-03-02 14:00:00')

mask = (df_dataset[timestamp] > start_date_train7) & (df_dataset[timestamp] <= end_date_train7)
df_train7 = df_dataset.loc[mask]

# pre - processamento
print(df_train7.shape)

list_periods_train = []
for pro in pre_process:
    df_train7,_,list_aux = ppd.drop_transitorio_desligado(df_train7,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_train = [*list_periods_train,*list_aux]

df_train7.reset_index(inplace=True, drop=True)

print(df_train7.shape)


eixoX_train7 = df_train7[timestamp]
df_train7 = df_train7.drop(timestamp,axis=1)

df_train = pd.concat([df_train, df_train7], ignore_index=True)
eixoX_train = pd.concat([eixoX_train, eixoX_train7], ignore_index=True)

df_train_list.append(df_train7)

## Período 8

In [ ]:
start_date_train8 = pd.to_datetime('2026-03-12 00:00:00') 
end_date_train8 = pd.to_datetime('2026-03-12 12:00:00')

mask = (df_dataset[timestamp] > start_date_train8) & (df_dataset[timestamp] <= end_date_train8)
df_train8 = df_dataset.loc[mask]

# pre - processamento
print(df_train8.shape)

list_periods_train = []
for pro in pre_process:
    df_train8,_,list_aux = ppd.drop_transitorio_desligado(df_train8,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_train = [*list_periods_train,*list_aux]

df_train8.reset_index(inplace=True, drop=True)

print(df_train8.shape)


eixoX_train8 = df_train8[timestamp]
df_train8 = df_train8.drop(timestamp,axis=1)

df_train = pd.concat([df_train, df_train8], ignore_index=True)
eixoX_train = pd.concat([eixoX_train, eixoX_train8], ignore_index=True)

df_train_list.append(df_train8)

## Criação do Modelo

In [ ]:
mscvae = MSCVAE()

gain = 1
epochs = 15
mscvae.fit(df_train_list, gain=gain, epochs=epochs, verbose=True)

print("--- Limiar de Anomalia ---")
print(mscvae.threshold)

# Seleção do período para testar o modelo

In [ ]:
start_date = pd.to_datetime("2026-03-12 00:00:00")
end_date = pd.to_datetime("2026-05-01 00:00:00")

mask = (df_dataset[timestamp] > start_date) & (df_dataset[timestamp] <= end_date)
df_test = df_dataset.loc[mask]

list_periods_nan = ppd.select_periods_nan(df_dataset,timestamp)

# pre - processamento
print(df_test.shape)

list_periods_test = []
for pro in pre_process:
    df_test,_,list_aux = ppd.drop_transitorio_desligado(df_test,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_test = [*list_periods_test,*list_aux]
    
print(df_test.shape)

df_test_aux = df_test.copy()
eixoX_test = df_test[timestamp]
df_test = df_test.drop(timestamp,axis=1)

df_test.reset_index(inplace=True, drop=True)

# Predição

In [ ]:
predictions = mscvae.predict(df_test, timestamps=eixoX_test)

## Gráfico de Detecção

In [ ]:
list_periods_test = ppd.merge_periods(list_periods_test)
fig_all_period = utils.graph_predict(predictions["phi"]/(mscvae.threshold), predictions["timestamp"], mscvae.threshold/(mscvae.threshold), None,start_date, end_date, list_periods=None, plot_anomalies=False)
fig_all_period.show()

# Contribuição

In [ ]:
contribution, df_reconstruction = mscvae.contribution(df_test, df_sistema)

In [ ]:
pd.DataFrame().from_dict(contribution).head(10)

## Gráfico Univariado

In [ ]:
start_date = pd.to_datetime("2026-03-12 00:00:00")
end_date = pd.to_datetime("2026-05-01 00:00:00")

mask = (df_dataset[timestamp] > start_date) & (df_dataset[timestamp] <= end_date)
df_aux = df_dataset.loc[mask]

eixoX_aux = df_aux[timestamp]
df_aux = df_aux.drop(timestamp,axis=1)

df_aux.reset_index(inplace=True, drop=True)
df_aux.head()

lista = ['392CT2150_MV']
fig = utils.graph_variables(df_aux, eixoX_aux,lista, df_projection=df_reconstruction)

fig.show()

In [ ]:
lista = ['392P2007A_TT']
fig = utils.graph_variables(df_aux, eixoX_aux,lista, df_projection=df_reconstruction)

fig.show()

In [ ]:
lista = ['392P2007.05HV.BM.LA']
fig = utils.graph_variables(df_aux, eixoX_aux,lista, df_projection=df_reconstruction)

fig.show()

In [ ]:
lista = ['392P2007.05HE2.BM.LA']
fig = utils.graph_variables(df_aux, eixoX_aux,lista, df_projection=df_reconstruction)

fig.show()

In [ ]:
# fig.write_html("output/Univar_"+process_name+".html")

# Gerar relatório de Validação

##### Não sendo necessário o relatório de validação não é necessário executar

In [ ]:
report = FuturaiReport(DIR_TEMPLATE,"Modelo 1 - ")

## Invertvalo de agrupamento de anomalias
anomaly_interval = 1440
## Caso seja necessário gerar relatório com a projeção das variáveis defina a variável abaixo como True
var_projection = False

report_name = report.build_validation(result,df_test_aux,df_dataset,df_train,mscvae,timestamp,df_sistema,fig_all_period,anomaly_interval,company_name,process_name,var_projection,pre_process=pre_process,type_graph=3)
report_name

# Gerar Pré Diagnóstico Executivo

In [ ]:
#Descomente se quiser habilitar as mensagens de log
logger = None
#logger = DiagnosticoExecutivo.prs_logger

# Não preencher esse dicionário
diary = {}

paradas_sem_agrupar, anomalias_sem_agrupar = DiagnosticoExecutivo.obter_paradas_anomalias(futurai=mscvae,
                                                                                          result=result,
                                                                                          df_desligado=df_desligado,
                                                                                          arquivo_anomalias=None,
                                                                                          arquivo_paradas=None,
                                                                                          coluna_motivo_parada=None,
                                                                                          prs_logger=logger)
DiagnosticoExecutivo.exportar_paradas(paradas_sem_agrupar, 'output/paradas_sem_agrupar.csv')
DiagnosticoExecutivo.exportar_anomalias(anomalias_sem_agrupar, 'output/anomalias_sem_agrupar.csv')

In [ ]:
#DiagnosticoExecutivo.print_paradas(paradas_sem_agrupar)
#DiagnosticoExecutivo.print_anomalias(anomalias_sem_agrupar)

In [ ]:
anomalias = DiagnosticoExecutivo.importar_anomalias('output/anomalias_sem_agrupar.csv')
anomalias  = DiagnosticoExecutivo.limitar_anomalias(anomalias,
                                                    intervalo_min_entre_anomalias_minutos=1440,
                                                    logger=logger,
                                                    diary=diary)
DiagnosticoExecutivo.exportar_anomalias(anomalias, 'output/anomalias_diagnostico.csv')

In [ ]:
# Se quiser adicionar o agrupamento, então descomentar as linhas abaixo
# anomalias = DiagnosticoExecutivo.importar_anomalias('output/anomalias_diagnostico.csv')
# DiagnosticoExecutivo.adicionar_agrupamento(anomalias, df_resultados, logger=None)
# DiagnosticoExecutivo.exportar_anomalias(anomalias, 'output/anomalias_diagnostico.csv')

In [ ]:
#DiagnosticoExecutivo.print_anomalias(anomalias)

In [ ]:
paradas = DiagnosticoExecutivo.importar_paradas('output/paradas_sem_agrupar.csv')
paradas = DiagnosticoExecutivo.resumir_paradas(paradas,
                                               intervalo_minimo_entre_paradas_minutos=240,
                                               duracao_minima_parada_minutos=60,
                                               agrupar_paradas_mesmo_dia=True,
                                               logger=logger,
                                               diary=diary)
DiagnosticoExecutivo.exportar_paradas(paradas, 'output/paradas_diagnostico.csv')

In [ ]:
#DiagnosticoExecutivo.print_paradas(paradas)

In [ ]:
anomalias = DiagnosticoExecutivo.importar_anomalias('output/anomalias_diagnostico.csv')
paradas = DiagnosticoExecutivo.importar_paradas('output/paradas_diagnostico.csv')
associacoes_sem_agrupar = DiagnosticoExecutivo.associar_anomalias_paradas(anomalias, paradas)
DiagnosticoExecutivo.exportar_associacoes(associacoes_sem_agrupar, 'output/associacoes_sem_agrupar.csv')

In [ ]:
associacoes = DiagnosticoExecutivo.importar_associacoes('output/associacoes_sem_agrupar.csv')
associacoes = DiagnosticoExecutivo.limitar_associacoes(associacoes,
                                                       intervalo_maximo_entre_anomalia_parada_horas=720,
                                                       limite_anomalias_por_associacao=8,
                                                       limites_paradas_por_associacao=1,
                                                       logger=logger,
                                                       diary=diary)
DiagnosticoExecutivo.exportar_associacoes(associacoes, 'output/associacoes_diagnostico.csv')

In [ ]:
DiagnosticoExecutivo.print_associacoes(associacoes)

In [ ]:
associacoes = DiagnosticoExecutivo.importar_associacoes('output/associacoes_diagnostico.csv')

In [ ]:
DiagnosticoExecutivo.adicionar_cores(associacoes,
                                     DiagnosticoExecutivo.COR_PADRAO_ANOMALIA,
                                     DiagnosticoExecutivo.COR_PADRAO_PARADA)


In [ ]:
## Caso seja necessário gerar diagnóstico com a projeção das variáveis defina a variável abaixo como True
var_projection = True

DiagnosticoExecutivo.adicionar_imagens(associacoes, mscvae, df_dataset,df_train,
                                       df_test_aux, df_sistema, timestamp,
                                       pre_process, var_projection, logger)

In [ ]:
diagnostico = DiagnosticoExecutivo.DiagnosisPresentation(process_name,
                                                         company_name,
                                                         template_name='template1.pptx',
                                                         logger=logger)

In [ ]:
corr_vars = select_var_ref(df_dataset,process_name,DIR_DATA)

#grafico_anomalia_inteiro = gerar_grafico_predicao_completo(df_test_aux, timestamp, futurai, pre_process, df_dataset)
grafico_anomalia_inteiro = fig_all_period

In [ ]:
diagnostico.build_pptx(associacoes,
                       correlation_variables=corr_vars,
                       columns_dropped=list_colomuns_drop,
                       prediction_graph=grafico_anomalia_inteiro,
                       reference_variables=pre_process,
                       dataset_filename=base_name,
                       timestamp_col=timestamp)

In [ ]:
#diagnostico.export_pptx("Nome personalizado.pptx", overwrite=False)
diagnostico.export_pptx(overwrite=False)

In [ ]:
#Salvar um csv com os parâmetros usados para limitar as anomalias, paradas e associacoes
#DiagnosticoExecutivo.exportar_diario(diary, 'output/prediagnostico_config.csv')

# Informações do modelo

In [ ]:
company_id, process_id, message, notebook_name, env, eventbridge, pp_period_good, type_task, atv_realizada, datafactory, author = utils.get_dados()

# Tracking do modelo e upload dos arquivos

In [ ]:
input_schedule = { "headers": "{'Cache-Control': 'no-cache','Ocp-Apim-Subscription-Key': '1c78eef199ca47318c82914f02890d93'}", "url": "https://api.suzano.com.br/pi-apis-p001/streams/", "company_id": company_id, "process_id": process_id }
input_schedule = json.dumps(input_schedule)

In [ ]:
filename_df_train, filename_df_sistema, filename_df_sistema_drop, model_name = model_dl.tracking_model_dl(
    company_id,
    process_id,
    df_train,
    eixoX_train,
    df_sistema,
    df_sistema_drop,
    timestamp,
    gain,
    pre_process,
    mscvae,
    pp_period_good,
    message,
    notebook_name,
    clickup_id,
    atv_realizada=atv_realizada,
    env=env
)

# Deploy

In [ ]:
model_type = "MSCVAE"
response = model_dl.deploy_model_dl(
    company_id, 
    process_id, 
    filename_df_train, 
    filename_df_sistema, 
    filename_df_sistema_drop, 
    timestamp, 
    pre_process, 
    pp_period_good, 
    type_task,
    message,
    author,
    model_type,
    model_name,
    datafactory=False,
    eventbridge=False,
    input_schedule="",
    env=env,
    drop_outliers=False,
    accuracy_model=None
)

response